# Authentication Walkthrough (Account Creation, Login, Logout)
This notebook explains the core ideas behind username/password authentication with JSON Web Tokens (JWT).
It is written for teammates who are new to JavaScript and authentication concepts.

## 1) Set Up Environment and Dependencies
We use Express for HTTP, bcrypt for password hashing, and jsonwebtoken for JWTs.
The code snippets below are simplified for learning and run as a single file (for example, `server.js`).
In your real app, these are split across routes, services, and middleware.

In [ ]:
// Example dependencies and basic app setup.
import express from 'express';
import bcrypt from 'bcryptjs';
import jwt from 'jsonwebtoken';

const app = express();
app.use(express.json());

// Store secrets in environment variables in real apps.
const JWT_SECRET = config.jwtSecret || 'dev-secret';
const TOKEN_EXPIRES_IN = '7d';

// Demo-only in-memory users (replace with a database).
const users = [];

## 2) Define the User Data Model and Password Hashing
A user record stores profile data and a **password hash** (never the raw password).
Hashing turns a password into a one-way string that cannot be reversed.
During login, we hash the provided password and compare it with the stored hash.

In [ ]:
// Simple user shape used for the demo.
function createUserRecord({ id, email, username, passwordHash }) {
  return {
    id,
    email,
    username,
    passwordHash,
    role: 'user',
  };
}

// Hash a password before saving.
async function hashPassword(password) {
  const saltRounds = 12;
  return bcrypt.hash(password, saltRounds);
}

// Compare a raw password with a saved hash.
async function verifyPassword(password, passwordHash) {
  return bcrypt.compare(password, passwordHash);
}

## 3) Create Account Registration Endpoint
Registration does four things:
1) Validate input.
2) Hash the password.
3) Store the user.
4) Return a success response (optionally with a token).

In [ ]:
app.post('/register', async (req, res) => {
  const { email, username, password } = req.body;

  if (!email || !username || !password) {
    return res.status(400).json({ message: 'Missing required fields' });
  }

  const existingUser = users.find((user) => user.email === email);
  if (existingUser) {
    return res.status(400).json({ message: 'Email already in use' });
  }

  const passwordHash = await hashPassword(password);
  const user = createUserRecord({
    id: String(Date.now()),
    email,
    username,
    passwordHash,
  });

  users.push(user);

  return res.status(201).json({
    message: 'Account created',
    user: { id: user.id, email: user.email, username: user.username },
  });
});

## 4) Implement Login and JWT Issuance
Login checks credentials and issues a JWT.
The JWT is a signed token that proves the user is authenticated.
Clients store the token (usually in memory or secure storage) and send it with each request.

In [ ]:
function signToken(user) {
  return jwt.sign({ sub: user.id, role: user.role }, JWT_SECRET, {
    expiresIn: TOKEN_EXPIRES_IN,
  });
}

app.post('/login', async (req, res) => {
  const { email, password } = req.body;

  if (!email || !password) {
    return res.status(400).json({ message: 'Missing required fields' });
  }

  const user = users.find((entry) => entry.email === email);
  if (!user) {
    return res.status(401).json({ message: 'Invalid credentials' });
  }

  const ok = await verifyPassword(password, user.passwordHash);
  if (!ok) {
    return res.status(401).json({ message: 'Invalid credentials' });
  }

  const token = signToken(user);
  return res.status(200).json({ token });
});

## 5) Protect Routes with Auth Middleware
Protected routes check the `Authorization` header for a bearer token.
If the token is valid, we allow the request and attach the user info to `req.user`.
If not, we return 401 (unauthorized).

In [ ]:
function requireAuth(req, res, next) {
  const header = req.headers.authorization || '';
  const [, token] = header.split(' ');

  if (!token) {
    return res.status(401).json({ message: 'Missing token' });
  }

  try {
    const payload = jwt.verify(token, JWT_SECRET);
    req.user = { id: payload.sub, role: payload.role };
    return next();
  } catch (err) {
    return res.status(401).json({ message: 'Invalid or expired token' });
  }
}

app.get('/me', requireAuth, (req, res) => {
  return res.status(200).json({ id: req.user.id, role: req.user.role });
});

## 6) Implement Logout and Token Invalidation
JWTs are stateless, so the server cannot "delete" a token.
Common options:
- Client deletes its stored token (simplest).
- Use short-lived tokens plus refresh tokens.
- Maintain a token blacklist (more server state).
Below is a simple blacklist demo for learning purposes.

In [ ]:
const tokenBlacklist = new Set();

app.post('/logout', requireAuth, (req, res) => {
  const header = req.headers.authorization || '';
  const [, token] = header.split(' ');

  if (token) {
    tokenBlacklist.add(token);
  }

  return res.status(204).send();
});

// Wrap requireAuth to check blacklist (demo only).
function requireAuthWithBlacklist(req, res, next) {
  const header = req.headers.authorization || '';
  const [, token] = header.split(' ');

  if (tokenBlacklist.has(token)) {
    return res.status(401).json({ message: 'Token is revoked' });
  }

  return requireAuth(req, res, next);
}

## 7) Test the Auth Flow with Example Requests
These examples use `curl` from a terminal.
Replace `http://localhost:3000` with your server URL.

```bash
# Register
curl -X POST http://localhost:3000/register \\
  -H "Content-Type: application/json" \\
  -d '{"email":"sam@example.com","username":"sam","password":"secret123"}'

# Login (copy the token from the response)
curl -X POST http://localhost:3000/login \\
  -H "Content-Type: application/json" \\
  -d '{"email":"sam@example.com","password":"secret123"}'

# Access a protected route
curl http://localhost:3000/me \\
  -H "Authorization: Bearer YOUR_TOKEN_HERE"

# Logout (server records the token as revoked in this demo)
curl -X POST http://localhost:3000/logout \\
  -H "Authorization: Bearer YOUR_TOKEN_HERE"
```

## Glossary
- **Hash**: One-way transformation of a password into a safe string.
- **JWT**: A signed token that proves a user is authenticated.
- **Bearer token**: The JWT sent in the `Authorization` header.
- **401 Unauthorized**: The server rejects the request due to missing or invalid credentials.